# Results analysis - tables and charts for the report

Reads every `results/*.json` written by `pipeline.ipynb`. No Spark needed.

1. Overview of every run
2. Per-stage breakdown and skew of one run
3. Before/after comparison of two runs (one per optimisation)
4. Optimisation summary table (report section 4)
5. Dataset scale chart and cluster-size speedup chart (report section 5)

In [ ]:
import glob
import json
import os

import matplotlib.pyplot as plt
import pandas as pd

RESULTS_DIR = "../results"
FIGURES_DIR = "../report/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

# Chart styling: one measured series (blue), reference lines in muted grey.
BLUE, GREY, INK = "#2a78d6", "#52514e", "#0b0b0b"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GREY, "axes.labelcolor": INK, "xtick.color": GREY, "ytick.color": GREY,
    "axes.grid": True, "grid.color": "#e5e4df", "grid.linewidth": 0.8,
    "lines.linewidth": 2, "lines.markersize": 8, "font.size": 10,
})


def load_results(results_dir=RESULTS_DIR):
    runs = {}
    for path in sorted(glob.glob(os.path.join(results_dir, "*.json"))):
        with open(path) as f:
            data = json.load(f)
        runs[data["metadata"]["run_name"]] = data
    return runs


def fmt_bytes(n):
    n = float(n or 0)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024:
            return f"{n:.1f}{unit}"
        n /= 1024
    return f"{n:.1f}PB"


results = load_results()
print(f"{len(results)} runs:", list(results))

## 1. Overview of every run

In [ ]:
def overview(results):
    rows = []
    for name, data in results.items():
        m = data["metadata"]
        row = {"run": name, "master": m["master"], "executors": m.get("num_executors"),
               "cores": m.get("total_cores"), "years": ",".join(map(str, m["years"])),
               "fraction": m.get("fraction"), "total_s": m["total_wall_time_s"]}
        row.update({f"{op}_s": r["wall_time_s"] for op, r in data["runs"].items()})
        rows.append(row)
    return pd.DataFrame(rows).set_index("run")


overview(results)

## 2. Per-operation totals and per-stage breakdown of one run

The brief asks for shuffle read/write bytes and task counts (on your
machine) and wall-clock and per-stage duration (on the cluster).
`skew_max_over_median` ≥ 10 marks a stage where one task takes 10× the
median.

In [ ]:
BYTE_COLS = ["input_bytes", "output_bytes", "shuffle_read_bytes", "shuffle_write_bytes",
             "memory_spilled_bytes", "disk_spilled_bytes"]


def op_table(run):
    df = pd.DataFrame({op: {k: v for k, v in r.items() if k != "stages"}
                       for op, r in results[run]["runs"].items()}).T
    for c in BYTE_COLS:
        df[c] = df[c].map(fmt_bytes)
    return df


def stage_table(run, op):
    df = pd.DataFrame(results[run]["runs"][op]["stages"]).set_index("stage_id")
    df["skewed"] = df["skew_max_over_median"].fillna(0) >= 10
    for c in [c for c in BYTE_COLS if c in df]:
        df[c] = df[c].map(fmt_bytes)
    return df.sort_values("duration_ms", ascending=False)


RUN = next(iter(results), None)   # pick the run to inspect
if RUN:
    display(op_table(RUN))

In [ ]:
if RUN:
    for op in results[RUN]["runs"]:
        print(op)
        display(stage_table(RUN, op).head(10))

## 3. Before/after comparison

Compare two runs, either for one operation or for the whole pipeline
(`op=None`). Each optimisation in the report needs this "before"
evidence, not just the "after".

In [ ]:
METRICS = ["wall_time_s", "num_stages", "num_tasks", "executor_run_time_ms",
           "executor_cpu_time_ms"] + BYTE_COLS


def totals(run, op=None):
    runs = results[run]["runs"]
    selected = [runs[op]] if op else list(runs.values())
    return {k: sum(r[k] for r in selected) for k in METRICS}


def compare(before, after, op=None):
    b, a = totals(before, op), totals(after, op)
    rows = []
    for k in METRICS:
        change = f"{(a[k] - b[k]) / b[k] * 100:+.1f}%" if b[k] else "-"
        show = fmt_bytes if k in BYTE_COLS else (lambda v: v)
        rows.append({"metric": k, before: show(b[k]), after: show(a[k]), "change": change})
    return pd.DataFrame(rows).set_index("metric")


# Example: compare("baseline", "opt-broadcast", op="o2")

## 4. Optimisation summary table (report section 4)

One entry per optimisation: `(name, before run, after run, operation or None)`.
Wall time is the headline; the second metric should be the one that
shows the bottleneck was removed (e.g. shuffle bytes for a broadcast join).

In [ ]:
OPTIMISATIONS = [
    # ("Broadcast zone lookup", "baseline", "opt1-broadcast", "o2", "shuffle_write_bytes"),
]

rows = []
for title, before, after, op, metric in OPTIMISATIONS:
    b, a = totals(before, op), totals(after, op)
    show = fmt_bytes if metric in BYTE_COLS else (lambda v: v)
    rows.append({
        "optimisation": title, "operation": op or "pipeline",
        "before_s": b["wall_time_s"], "after_s": a["wall_time_s"],
        "speedup": round(b["wall_time_s"] / a["wall_time_s"], 2) if a["wall_time_s"] else None,
        "metric": metric, "metric_before": show(b[metric]), "metric_after": show(a[metric]),
    })
summary = pd.DataFrame(rows)
summary
# summary.to_latex(index=False)  -> paste into the report (needs jinja2)

## 5a. Dataset scale

Run the optimised pipeline on a 10-20% sample and then on the full
dataset (`-p FRACTION 0.1`, `-p FRACTION 0.2`, ...), then map each
fraction to its run name below. The dashed line is linear scaling from
the smallest fraction: points above it mean worse-than-linear scaling,
points below it mean sub-linear.

In [ ]:
SCALE_RUNS = {
    # 0.1: "scale-0.1",
    # 0.2: "scale-0.2",
    # 1.0: "scale-1.0",
}

if SCALE_RUNS:
    fr = sorted(SCALE_RUNS)
    t = [results[SCALE_RUNS[f]]["metadata"]["total_wall_time_s"] for f in fr]
    fig, ax = plt.subplots(figsize=(5, 3.2))
    ax.plot(fr, [t[0] * f / fr[0] for f in fr], "--", color=GREY, linewidth=1.5, label="linear scaling")
    ax.plot(fr, t, "-o", color=BLUE, label="measured")
    ax.set_xlabel("Dataset fraction")
    ax.set_ylabel("Wall-clock time (s)")
    ax.set_ylim(bottom=0)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, "scale_fraction.pdf"))
    plt.show()
    display(pd.DataFrame({"fraction": fr, "time_s": t}))

## 5b. Cluster size: speedup and efficiency

Run the same configuration with 2 workers and with the full allocation
(`docker compose up -d --scale spark-worker=N`), then map each worker
count to its run name. Speedup is relative to the smallest worker count:
S(n) = T(n₀) / T(n), efficiency E(n) = S(n) / (n / n₀).

In [ ]:
WORKER_RUNS = {
    # 2: "cluster-2w",
    # 4: "cluster-4w",
}

if WORKER_RUNS:
    n = sorted(WORKER_RUNS)
    t = [results[WORKER_RUNS[k]]["metadata"]["total_wall_time_s"] for k in n]
    speedup = [t[0] / x for x in t]
    efficiency = [s / (k / n[0]) for s, k in zip(speedup, n)]
    display(pd.DataFrame({"workers": n, "time_s": t, "speedup": speedup,
                          "efficiency": efficiency}).round(2))

    fig, ax = plt.subplots(figsize=(5, 3.2))
    ax.plot(n, [k / n[0] for k in n], "--", color=GREY, linewidth=1.5, label="ideal (linear)")
    ax.plot(n, speedup, "-o", color=BLUE, label="measured")
    ax.set_xlabel("Workers")
    ax.set_ylabel(f"Speedup vs {n[0]} workers")
    ax.set_xticks(n)
    ax.set_ylim(bottom=0)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, "speedup_workers.pdf"))
    plt.show()